In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

In order to display HTML in the notebook, we load the library `jupyter.notebook`, which provides the function `Jupyter_notebook.display`.

In [ ]:
#require "jupyter.notebook";;

# The Bridge and the Torch

In the darkness of the night, a group of four individuals encounters a river. A slender bridge stretches before them, capable of accommodating just two people simultaneously. Equipped with a single torch, they must rely on its flickering light to navigate the bridge. Each person possesses a distinct crossing time: Ariela takes 1 minute, Brian takes 2 minutes, Charly takes 5 minutes, and Dumpy takes 8 minutes. It is crucial to note that when two people cross together, they must synchronize their steps with the slower individual's pace. Given the torch's limited lifespan of 15 minutes, the pressing question arises: can all four individuals successfully traverse the bridge?

In [ ]:
import_file "../z3_helpers.ml"

We need the following variables to encode the problem:
* `A` equals `1` if Ariela is on the left shore,
* `B` equals `1` if Brian  is on the left shore,
* `C` equals `1` if Charly is on the left shore,
* `D` equals `1` if Dumpy  is on the left shore,
* `T` equals the time that has passed.

As the names of <samp>OCaml</samp> variables have to start with a lower case letter, the parameters of the following functions are called `a`, `b`, `c`, `d`, and `t`.  The formulas are written as strings, which use the names `A`, `B`, `C`, `D`, and `T`.  The optional argument `~vars` of the function `constraints` binds these names to the parameters, e.g. `constraints ~vars:["A", a] ["A = 1"]` returns a list containing the formula $\texttt{A} = 1$, where `A` stands for the parameter `a`.

Replace every occurrence of `failwith "your code here"` with your code.  The function `failwith` raises an exception.  As it can be used in place of an expression of any type, it serves as a placeholder.

The function `start` takes five `Z3` variables `a`, `b`, `c`, `d`, and `t` as arguments.  It returns a list of constraints that specify that everybody is on the left side of the river.
Furthermore, no time has passed.
$$ \texttt{A} = 1 \wedge \texttt{B} = 1 \wedge \texttt{C} = 1 \wedge \texttt{D} = 1 \wedge \texttt{T} = 0 $$

In [ ]:
let start (a : Z3.Expr.expr) (b : Z3.Expr.expr) (c : Z3.Expr.expr) (d : Z3.Expr.expr)
          (t : Z3.Expr.expr) : Z3.Expr.expr list =
  failwith "your code here"

The function `goal` takes the same arguments as `start`.  It returns a list of constraints that specify that everybody is on the right side of the river.
The time must be less or equal than $15$:
$$ \texttt{A} = 0 \wedge \texttt{B} = 0 \wedge \texttt{C} = 0 \wedge \texttt{D} = 0 \wedge \texttt{T} \leq 15 $$

In [ ]:
let goal (a : Z3.Expr.expr) (b : Z3.Expr.expr) (c : Z3.Expr.expr) (d : Z3.Expr.expr)
         (t : Z3.Expr.expr) : Z3.Expr.expr list =
  failwith "your code here"

The function `torch_ok` describes one crossing of the bridge from left to right.  It takes ten arguments:
* `a` is the number of Arielas on the left side before the crossing,
   while `ax` is the number of Arielas on the left side after the crossing.
* Dito for `b`, `bx`, $\cdots$, `d`, `dx`.
* `t` is the time before the crossing and `tx` is the time after the crossing.

It returns a list of constraints that ensure that
 * people go from left to right,
 * at least one person crosses the bridge,
 * at most two persons cross the bridge,
 * the variable `tx` is incremented by the right amount.

In [ ]:
let torch_ok (a : Z3.Expr.expr) (b : Z3.Expr.expr) (c : Z3.Expr.expr) (d : Z3.Expr.expr) (t : Z3.Expr.expr)
             (ax : Z3.Expr.expr) (bx : Z3.Expr.expr) (cx : Z3.Expr.expr) (dx : Z3.Expr.expr) (tx : Z3.Expr.expr)
             : Z3.Expr.expr list =
  failwith "your code here"

The function `transition` takes the same ten arguments as `torch_ok` and additionally the number `i` of the crossing.  The first crossing has `i = 0`.  It returns a list of constraints that describe the crossing:  The crossings with an even number go from left to right, the others go from right to left.  A crossing from right to left is described by swapping the roles of the variables before and after the crossing, except for the time.

In [ ]:
let transition (a : Z3.Expr.expr) (b : Z3.Expr.expr) (c : Z3.Expr.expr) (d : Z3.Expr.expr) (t : Z3.Expr.expr)
               (ax : Z3.Expr.expr) (bx : Z3.Expr.expr) (cx : Z3.Expr.expr) (dx : Z3.Expr.expr) (tx : Z3.Expr.expr)
               (i : int) : Z3.Expr.expr list =
  if i mod 2 = 0 then
    failwith "your code here"
  else
    failwith "your code here"

The function `bridge_csp` takes one argument:
- `n` is the number of crossings.

It tries to solve the problem using `n` crossings of the bridge.
If this is possible, `Some solution` is returned, where `solution` is a map that maps the variables
to their values.  Otherwise, `None` is returned.

In [ ]:
let bridge_csp (n : int) : (string, int) RMap.t option =
  let solver = new_solver () in
  let vars x = Array.init (n + 1) (fun i -> int_const (Printf.sprintf "%s%d" x i)) in
  let as_ = vars "A" and bs = vars "B" and cs = vars "C" and ds = vars "D" and ts = vars "T" in
  let bounds i =
    constraints ~vars:["A", as_.(i); "B", bs.(i); "C", cs.(i); "D", ds.(i); "T", ts.(i)]
      ["0 <= A <= 1"; "0 <= B <= 1"; "0 <= C <= 1"; "0 <= D <= 1"; "T >= 0"]
  in
  Z3.Solver.add solver (start as_.(0) bs.(0) cs.(0) ds.(0) ts.(0));
  Z3.Solver.add solver (goal  as_.(n) bs.(n) cs.(n) ds.(n) ts.(n));
  for i = 0 to n - 1 do
    Z3.Solver.add solver (transition as_.(i) bs.(i) cs.(i) ds.(i) ts.(i)
                                     as_.(i + 1) bs.(i + 1) cs.(i + 1) ds.(i + 1) ts.(i + 1) i);
    Z3.Solver.add solver (bounds i)
  done;
  (* Bounds for the final step *)
  Z3.Solver.add solver (bounds n);
  match Z3.Solver.check solver [] with
  | Z3.Solver.SATISFIABLE ->
      let model = solver_model solver in
      Some (RMap.of_list
              (List.concat
                 (List.init (n + 1)
                    (fun i ->
                       List.map (fun (x, vs) -> (Printf.sprintf "%s%d" x i, eval_int model vs.(i)))
                                [("A", as_); ("B", bs); ("C", cs); ("D", ds); ("T", ts)]))))
  | _ -> None

The function `find_solution` takes the argument `()`.  It tries an increasing odd number of crossings until a solution is found.  The result is the pair `(n, solution)`.

In [ ]:
let find_solution () : int * (string, int) RMap.t =
  let rec try_crossings n =
    Printf.printf "%d\n" n;
    match bridge_csp n with
    | Some solution -> (n, solution)
    | None          -> try_crossings (n + 2)
  in
  try_crossings 1

In [ ]:
let (n, solution) = time "Bridge Solving" find_solution

The function `show_solution` takes two arguments:
- `sol` is a solution, i.e. a map from the names of the variables to their values,
- `n` is the number of crossings.

It prints the states and the persons crossing the bridge for every crossing.  Every person needs two columns of space.

In [ ]:
let repeat (s : string) (k : int) : string =
  String.concat "" (List.init k (fun _ -> s))

let show_solution (sol : (string, int) RMap.t) (n : int) : unit =
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) sol in
  let persons pa pb pc pd =
    repeat "🏃‍♀️" pa ^ repeat "🏃🏽‍♂️" pb ^ repeat "🚶🏽‍♂️" pc ^ repeat "👨‍🦽" pd
  in
  for i = 0 to n do
    let a = get "A" i and b = get "B" i and c = get "C" i and d = get "D" i in
    let left  = persons a b c d in
    let right = persons (1 - a) (1 - b) (1 - c) (1 - d) in
    print_endline (left ^ String.make (2 * (4 - (a + b + c + d)) + 22) ' ' ^ right);
    Printf.printf "🕰️%d\n" (get "T" i);
    if i < n then begin
      let a' = get "A" (i + 1) and b' = get "B" (i + 1) and c' = get "C" (i + 1) and d' = get "D" (i + 1) in
      if i mod 2 = 0 then
        print_endline (String.make 24 ' ' ^ ">> " ^ persons (a - a') (b - b') (c - c') (d - d') ^ " >>")
      else
        print_endline (String.make 24 ' ' ^ "<< " ^ persons (a' - a) (b' - b) (c' - c) (d' - d) ^ " <<")
    end
  done

In [ ]:
show_solution solution n

The function `show_animated_solution` takes the same arguments as `show_solution`.  It displays an animation of the solution.  As in the notebook `05-Missionaries-STS.ipynb`, the HTML code is stored in the quoted string `template`, which contains the placeholders `__UID__`, `__N__`, and `__STATES__`.  The function `replace_all s pattern by` replaces every occurrence of the string `pattern` in `s` by the string `by`.

In [ ]:
let replace_all (s : string) (pattern : string) (by : string) : string =
  let n = String.length s and k = String.length pattern in
  let buffer = Buffer.create n in
  let rec loop i =
    if i > n - k then
      Buffer.add_string buffer (String.sub s i (n - i))
    else if String.sub s i k = pattern then begin
      Buffer.add_string buffer by;
      loop (i + k)
    end else begin
      Buffer.add_char buffer s.[i];
      loop (i + 1)
    end
  in
  loop 0;
  Buffer.contents buffer

let template = {|
    <div style="font-family: sans-serif; max-width: 600px; margin: 0 auto; text-align: center;">
        <div style="margin-bottom: 15px;">
            <button id="btn-reset-__UID__" style="padding: 8px 16px; cursor: pointer;">Reset</button>
            <button id="btn-prev-__UID__" style="padding: 8px 16px; cursor: pointer;">Previous Step</button>
            <button id="btn-next-__UID__" style="padding: 8px 16px; cursor: pointer; font-weight: bold;">Next Step</button>
            <span id="label-step-__UID__" style="margin-left: 15px; font-weight: bold; font-size: 16px;">Step 0 of __N__</span>
        </div>
        <div style="margin-bottom: 10px; font-size: 18px; font-weight: bold; color: #d32f2f;">Time Elapsed: <span id="timer-__UID__">0</span> mins</div>
        <svg width="600" height="300" style="background-color: #0b1120; border: 1px solid #333; border-radius: 8px;">
            <rect x="0" y="0" width="150" height="300" fill="#1e3a29" />
            <rect x="450" y="0" width="150" height="300" fill="#1e3a29" />
            <rect x="150" y="130" width="300" height="40" fill="#5c4033" stroke="#3b261e" stroke-width="4" />
            
            <text x="75" y="30" font-size="14" font-weight="bold" fill="#a5d6a7" text-anchor="middle">Left Shore</text>
            <text x="525" y="30" font-size="14" font-weight="bold" fill="#a5d6a7" text-anchor="middle">Right Shore</text>
            
            <text id="west-a-__UID__" x="75" y="100" font-size="24" text-anchor="middle"></text>
            <text id="west-b-__UID__" x="75" y="140" font-size="24" text-anchor="middle"></text>
            <text id="west-c-__UID__" x="75" y="180" font-size="24" text-anchor="middle"></text>
            <text id="west-d-__UID__" x="75" y="220" font-size="24" text-anchor="middle"></text>

            <text id="east-a-__UID__" x="525" y="100" font-size="24" text-anchor="middle"></text>
            <text id="east-b-__UID__" x="525" y="140" font-size="24" text-anchor="middle"></text>
            <text id="east-c-__UID__" x="525" y="180" font-size="24" text-anchor="middle"></text>
            <text id="east-d-__UID__" x="525" y="220" font-size="24" text-anchor="middle"></text>

            <g id="group-__UID__" style="transition: transform 1.2s ease-in-out;">
                <text x="200" y="110" font-size="28" text-anchor="middle">🔦</text>
                <text id="bridge-p-__UID__" x="200" y="160" font-size="24" text-anchor="middle"></text>
            </g>
        </svg>
    </div>
    <script>
    (function() {
        const states = __STATES__;
        let currentStep = 0;
        let isAnimating = false;
        
        const group = document.getElementById('group-__UID__');
        const wA = document.getElementById('west-a-__UID__');
        const wB = document.getElementById('west-b-__UID__');
        const wC = document.getElementById('west-c-__UID__');
        const wD = document.getElementById('west-d-__UID__');
        const eA = document.getElementById('east-a-__UID__');
        const eB = document.getElementById('east-b-__UID__');
        const eC = document.getElementById('east-c-__UID__');
        const eD = document.getElementById('east-d-__UID__');
        const bridgeP = document.getElementById('bridge-p-__UID__');
        const stepLabel = document.getElementById('label-step-__UID__');
        const timerLabel = document.getElementById('timer-__UID__');
        
        function renderStaticState(step) {
            const s = states[step];
            wA.textContent = '🏃‍♀️'.repeat(s.A);
            wB.textContent = '🏃🏽‍♂️'.repeat(s.B);
            wC.textContent = '🚶🏽‍♂️'.repeat(s.C);
            wD.textContent = '👨‍🦽'.repeat(s.D);
            eA.textContent = '🏃‍♀️'.repeat(1 - s.A);
            eB.textContent = '🏃🏽‍♂️'.repeat(1 - s.B);
            eC.textContent = '🚶🏽‍♂️'.repeat(1 - s.C);
            eD.textContent = '👨‍🦽'.repeat(1 - s.D);
            bridgeP.textContent = '';
            
            group.style.transform = s.torchLeft ? 'translateX(0px)' : 'translateX(200px)';
            stepLabel.textContent = 'Step ' + step + ' of __N__';
            timerLabel.textContent = s.T;
        }

        async function animateTransition(fromStep, toStep) {
            if (isAnimating) return;
            isAnimating = true;
            
            const s0 = states[fromStep];
            const s1 = states[toStep];
            
            const aPass = Math.abs(s0.A - s1.A);
            const bPass = Math.abs(s0.B - s1.B);
            const cPass = Math.abs(s0.C - s1.C);
            const dPass = Math.abs(s0.D - s1.D);
            
            if (s0.torchLeft) { 
                wA.textContent = '🏃‍♀️'.repeat(s0.A - aPass);
                wB.textContent = '🏃🏽‍♂️'.repeat(s0.B - bPass);
                wC.textContent = '🚶🏽‍♂️'.repeat(s0.C - cPass);
                wD.textContent = '👨‍🦽'.repeat(s0.D - dPass);
            } else { 
                eA.textContent = '🏃‍♀️'.repeat((1 - s0.A) - aPass);
                eB.textContent = '🏃🏽‍♂️'.repeat((1 - s0.B) - bPass);
                eC.textContent = '🚶🏽‍♂️'.repeat((1 - s0.C) - cPass);
                eD.textContent = '👨‍🦽'.repeat((1 - s0.D) - dPass);
            }
            bridgeP.textContent = '🏃‍♀️'.repeat(aPass) + '🏃🏽‍♂️'.repeat(bPass) + '🚶🏽‍♂️'.repeat(cPass) + '👨‍🦽'.repeat(dPass);
            
            group.style.transform = s1.torchLeft ? 'translateX(0px)' : 'translateX(200px)';
            
            await new Promise(r => setTimeout(r, 1200));
            
            currentStep = toStep;
            renderStaticState(currentStep);
            isAnimating = false;
        }

        document.getElementById('btn-next-__UID__').addEventListener('click', () => {
            if (currentStep < states.length - 1) animateTransition(currentStep, currentStep + 1);
        });
        document.getElementById('btn-prev-__UID__').addEventListener('click', () => {
            if (currentStep > 0) animateTransition(currentStep, currentStep - 1);
        });
        document.getElementById('btn-reset-__UID__').addEventListener('click', () => {
            if (!isAnimating) {
                currentStep = 0;
                renderStaticState(0);
            }
        });

        renderStaticState(0);
    })();
    </script>
    |}

let show_animated_solution (sol : (string, int) RMap.t) (n : int) : unit =
  let get x i = RMap.find (Printf.sprintf "%s%d" x i) sol in
  let states =
    "[" ^ String.concat ", "
            (List.init (n + 1)
               (fun i -> Printf.sprintf "{\"A\": %d, \"B\": %d, \"C\": %d, \"D\": %d, \"T\": %d, \"torchLeft\": %b}"
                           (get "A" i) (get "B" i) (get "C" i) (get "D" i) (get "T" i) (i mod 2 = 0)))
    ^ "]"
  in
  let uid  = Printf.sprintf "%07x" (Random.bits ()) in
  let html = replace_all (replace_all (replace_all template "__UID__" uid) "__N__" (string_of_int n))
                         "__STATES__" states in
  ignore (Jupyter_notebook.display "text/html" html)

In [ ]:
show_animated_solution solution n